# Advanced Python Assignment Expressions (`:=`)
## Problems, Solutions, Tests, Pitfalls, and Best Practices

This notebook is a problem-driven study of **assignment expressions**, introduced in Python 3.8 and commonly called the **walrus operator**.

An assignment expression:

```python
name := expression
```

evaluates `expression`, binds the result to `name`, and returns that same result as the value of the surrounding expression.

The main engineering use case is to **compute a value once and reuse it immediately inside a condition, comprehension, generator expression, or loop**.

> Run this notebook with Python 3.8 or newer. It uses only the Python standard library.

## Learning objectives

By the end of the notebook, you should be able to:

- distinguish ordinary assignment (`=`) from assignment expressions (`:=`);
- use `:=` safely in `if`, `while`, comprehensions, and generator expressions;
- eliminate repeated function calls and parsing;
- reason about evaluation order and short-circuiting;
- recognize precedence, parenthesization, and scope traps;
- preserve falsy-but-valid values;
- reject clever but unreadable uses;
- test refactorings for semantic equivalence;
- build a realistic streaming-data pipeline.

In [1]:
import sys

assert sys.version_info >= (3, 8), "This notebook requires Python 3.8+."
print(sys.version)

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


## Compact reference

### Ordinary assignment

```python
result = expensive_call()
```

This is a statement.

### Assignment expression

```python
if (result := expensive_call()) is not None:
    use(result)
```

This computes once, tests the value, and makes it available in the body.

### Best-practice rule

Use `:=` when it removes meaningful duplication and keeps the code easier to read. Do **not** use it merely to reduce line count.

In [2]:
value_returned = (x := 10 + 20)

assert value_returned == 30
assert x == 30
print(value_returned, x)

30 30


## Syntax and target restrictions

A bare assignment expression normally needs parentheses when used as a standalone expression:

```python
(x := 3)
```

The target must be a simple name. Attribute and item targets are invalid:

```python
obj.attr := 1
items[0] := 1
```

The next cell checks syntax safely with `compile`.

In [3]:
syntax_examples = {
    "valid_parenthesized": "(x := 1 + 2)",
    "invalid_bare_statement": "x := 1 + 2",
    "invalid_attribute_target": "(obj.attr := 10)",
    "invalid_subscript_target": "(items[0] := 10)",
}

for label, source in syntax_examples.items():
    try:
        compile(source, f"<{label}>", "exec")
    except SyntaxError as exc:
        print(f"{label:28} -> SyntaxError: {exc.msg}")
    else:
        print(f"{label:28} -> valid")

valid_parenthesized          -> valid
invalid_bare_statement       -> SyntaxError: invalid syntax
invalid_attribute_target     -> SyntaxError: cannot use assignment expressions with attribute
invalid_subscript_target     -> SyntaxError: cannot use assignment expressions with subscript


## Best-practice checklist

Before using `:=`, ask:

1. Is the computed value genuinely reused?
2. Is the binding close to its use?
3. Is the variable name descriptive?
4. Does the expression remain easy to scan?
5. Are side effects obvious and intentional?
6. Would an ordinary assignment be clearer?
7. Have evaluation order and short-circuiting been tested?

# Problem 1 — Eliminate duplicate expensive calls

A program calculates a score for each input. The original comprehension calls the scoring function once in the filter and again in the output expression.

Tasks:

1. prove that the original performs duplicate calls;
2. refactor it with `:=`;
3. preserve the exact output;
4. verify the call-count improvement.

In [4]:
from collections import Counter

call_counts = Counter()

def score_record(record: dict) -> int:
    # Pretend this function is expensive or performs I/O.
    call_counts[record["id"]] += 1
    return record["quality"] * 3 - record["penalty"] * 2

records = [
    {"id": "A", "quality": 7, "penalty": 1},
    {"id": "B", "quality": 2, "penalty": 4},
    {"id": "C", "quality": 9, "penalty": 2},
    {"id": "D", "quality": 5, "penalty": 5},
]
threshold = 10

### Starter version

In [5]:
call_counts.clear()

duplicate_work = [
    {"id": record["id"], "score": score_record(record)}
    for record in records
    if score_record(record) >= threshold
]

print("Result:", duplicate_work)
print("Calls:", dict(call_counts))

Result: [{'id': 'A', 'score': 19}, {'id': 'C', 'score': 23}]
Calls: {'A': 2, 'B': 1, 'C': 2, 'D': 1}


### Solution

In [6]:
call_counts.clear()

single_evaluation = [
    {"id": record["id"], "score": score}
    for record in records
    if (score := score_record(record)) >= threshold
]

print("Result:", single_evaluation)
print("Calls:", dict(call_counts))

assert single_evaluation == duplicate_work
assert sum(call_counts.values()) == len(records)
assert all(count == 1 for count in call_counts.values())

Result: [{'id': 'A', 'score': 19}, {'id': 'C', 'score': 23}]
Calls: {'A': 1, 'B': 1, 'C': 1, 'D': 1}


### Why this is a good use

`score` is needed by both the filter and the output. The assignment belongs in the `if` clause because comprehension evaluation order is:

1. iterate;
2. evaluate filters;
3. evaluate the output expression.

# Problem 2 — Parse log lines exactly once

Extract timestamp, severity, and message only from valid log lines whose severity is `ERROR` or `CRITICAL`.

Constraints:

- call `fullmatch` no more than once per line;
- discard malformed lines;
- normalize severity to uppercase.

In [7]:
import re

log_pattern = re.compile(
    r"(?P<timestamp>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}) "
    r"\[(?P<severity>[A-Za-z]+)\] "
    r"(?P<message>.+)"
)

log_lines = [
    "2026-07-29T09:10:00 [INFO] service started",
    "2026-07-29T09:12:10 [error] database unavailable",
    "badly formatted line",
    "2026-07-29T09:13:22 [CRITICAL] queue corruption detected",
    "2026-07-29T09:14:03 [WARNING] latency elevated",
]

### Solution

In [8]:
serious_events = [
    {
        "timestamp": match["timestamp"],
        "severity": match["severity"].upper(),
        "message": match["message"],
    }
    for line in log_lines
    if (match := log_pattern.fullmatch(line))
    and match["severity"].upper() in {"ERROR", "CRITICAL"}
]

serious_events

[{'timestamp': '2026-07-29T09:12:10',
  'severity': 'ERROR',
  'message': 'database unavailable'},
 {'timestamp': '2026-07-29T09:13:22',
  'severity': 'CRITICAL',
  'message': 'queue corruption detected'}]

In [9]:
assert serious_events == [
    {
        "timestamp": "2026-07-29T09:12:10",
        "severity": "ERROR",
        "message": "database unavailable",
    },
    {
        "timestamp": "2026-07-29T09:13:22",
        "severity": "CRITICAL",
        "message": "queue corruption detected",
    },
]

Because `and` short-circuits, `match["severity"]` is evaluated only when `fullmatch` returned a match object.

# Problem 3 — Read a stream in chunks

Read all bytes from an in-memory stream in fixed-size chunks.

Requirements:

- no infinite loop;
- no duplicated `read` call;
- preserve chunk boundaries;
- stop when `read` returns `b""`.

In [10]:
from io import BytesIO

payload = b"assignment-expressions-are-useful"
stream = BytesIO(payload)
chunk_size = 7

### Conventional solution

In [11]:
stream.seek(0)
chunks_conventional = []

while True:
    chunk = stream.read(chunk_size)
    if not chunk:
        break
    chunks_conventional.append(chunk)

chunks_conventional

[b'assignm', b'ent-exp', b'ression', b's-are-u', b'seful']

### Assignment-expression solution

In [12]:
stream.seek(0)
chunks_walrus = []

while chunk := stream.read(chunk_size):
    chunks_walrus.append(chunk)

print(chunks_walrus)
assert chunks_walrus == chunks_conventional
assert b"".join(chunks_walrus) == payload

[b'assignm', b'ent-exp', b'ression', b's-are-u', b'seful']


This is a canonical use of `:=`: fetch the next value, test the stopping condition, and make the fetched value available in the loop body.

# Problem 4 — Consume an iterator until a sentinel

Process measurements until either the iterator is exhausted or `None` is encountered.

Do not lose valid zero values, and do not let `StopIteration` escape.

In [13]:
measurements = iter([12.5, 0.0, 13.2, 11.8, None, 99.9])
_END = object()
accepted = []

### Solution

In [14]:
while (measurement := next(measurements, _END)) is not _END and measurement is not None:
    accepted.append(measurement)

accepted

[12.5, 0.0, 13.2, 11.8]

In [15]:
assert accepted == [12.5, 0.0, 13.2, 11.8]

### Common bug

```python
while measurement := next(iterator, None):
    ...
```

This incorrectly stops for valid falsy values such as `0`, `0.0`, `""`, and empty containers. Use an explicit sentinel when falsy values are meaningful.

# Problem 5 — JSON validation pipeline

Keep only JSON objects that:

- parse successfully;
- have `"active": true`;
- have a numeric `"score"` of at least 80.

A failed parse must not terminate the pipeline.

In [16]:
import json
from typing import Any, Optional

raw_rows = [
    '{"user": "Ana", "active": true, "score": 92}',
    '{"user": "Boris", "active": false, "score": 95}',
    'not JSON',
    '{"user": "Chen", "active": true, "score": 79}',
    '{"user": "Dina", "active": true, "score": 88.5}',
    '["not", "an", "object"]',
]

def parse_json_object(text: str) -> Optional[dict[str, Any]]:
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

### Solution

In [17]:
qualified_users = [
    {"user": obj["user"], "score": obj["score"]}
    for raw in raw_rows
    if (obj := parse_json_object(raw)) is not None
    and obj.get("active") is True
    and isinstance(obj.get("score"), (int, float))
    and not isinstance(obj.get("score"), bool)
    and obj["score"] >= 80
]

qualified_users

[{'user': 'Ana', 'score': 92}, {'user': 'Dina', 'score': 88.5}]

In [18]:
assert qualified_users == [
    {"user": "Ana", "score": 92},
    {"user": "Dina", "score": 88.5},
]

The safe parser handles exception control flow; the assignment expression handles reuse. `:=` does not catch exceptions.

# Problem 6 — Short-circuiting and side effects

Determine which functions are called in:

```python
if (token := fetch_token()) and (profile := fetch_profile(token)):
    ...
```

Test three cases: missing token, missing profile, and success.

In [19]:
events = []

def make_fetchers(token_value, profile_value):
    def fetch_token():
        events.append("fetch_token")
        return token_value

    def fetch_profile(token):
        events.append(f"fetch_profile:{token}")
        return profile_value

    return fetch_token, fetch_profile

def run_auth_case(token_value, profile_value):
    events.clear()
    fetch_token, fetch_profile = make_fetchers(token_value, profile_value)

    if (token := fetch_token()) and (profile := fetch_profile(token)):
        outcome = f"welcome {profile['name']}"
    else:
        outcome = "authentication failed"

    return outcome, list(events)

### Solution and tests

In [20]:
case_1 = run_auth_case("", {"name": "Ignored"})
case_2 = run_auth_case("abc", None)
case_3 = run_auth_case("abc", {"name": "Mira"})

print(case_1)
print(case_2)
print(case_3)

assert case_1 == ("authentication failed", ["fetch_token"])
assert case_2 == ("authentication failed", ["fetch_token", "fetch_profile:abc"])
assert case_3 == ("welcome Mira", ["fetch_token", "fetch_profile:abc"])

('authentication failed', ['fetch_token'])
('authentication failed', ['fetch_token', 'fetch_profile:abc'])
('welcome Mira', ['fetch_token', 'fetch_profile:abc'])


Assignment expressions preserve normal left-to-right Boolean short-circuit behavior.

# Problem 7 — Correct placement in a comprehension

This idea is wrong because the filter uses `normalized` before the output expression assigns it:

```python
[
    (normalized := text.strip().lower())
    for text in values
    if normalized
]
```

Normalize once, remove blank strings, and return normalized values.

In [21]:
values = ["  Alpha ", " ", "BETA", "\tGamma\n", "", " delta "]

### Solution

In [22]:
normalized_values = [
    normalized
    for text in values
    if (normalized := text.strip().lower())
]

normalized_values

['alpha', 'beta', 'gamma', 'delta']

In [23]:
assert normalized_values == ["alpha", "beta", "gamma", "delta"]

A comprehension evaluates its filters before its output expression. Put the assignment where it is first needed.

# Problem 8 — Scope behavior in comprehensions

Test whether a name assigned by `:=` inside a comprehension remains available afterward. Compare it with the iteration variable.

In [24]:
globals().pop("last_square", None)
globals().pop("number", None)

squares = [
    last_square
    for number in range(6)
    if (last_square := number * number) % 2 == 1
]

print("squares:", squares)
print("last_square after comprehension:", last_square)
print("number leaked?", "number" in globals())

squares: [1, 9, 25]
last_square after comprehension: 25
number leaked? False


In [25]:
assert squares == [1, 9, 25]
assert last_square == 25
assert "number" not in globals()

The comprehension iteration variable does not leak in Python 3. A name bound by an assignment expression is bound in the containing scope. This can surprise readers, so do not rely on it as an implicit output mechanism.

# Problem 9 — Parentheses and comparison precedence

Predict the values and types in:

```python
result_1 = (a := 5 > 3)
result_2 = ((b := 5) > 3)
```

In [26]:
result_1 = (a := 5 > 3)
result_2 = ((b := 5) > 3)

print("a:", a, type(a))
print("b:", b, type(b))
print("result_1:", result_1)
print("result_2:", result_2)

a: True <class 'bool'>
b: 5 <class 'int'>
result_1: True
result_2: True


In [27]:
assert a is True
assert b == 5
assert result_1 is True
assert result_2 is True

`(a := 5 > 3)` assigns the comparison result (`True`). `((b := 5) > 3)` assigns `5` and then compares it. Parentheses should make the intended assigned value obvious.

# Problem 10 — Transform, validate, and aggregate CSV rows

Parse product transactions. Keep rows where:

- quantity is a positive integer;
- unit price is a positive decimal;
- the line total is at least 100.

Avoid repeated conversions and multiplication.

In [28]:
from decimal import Decimal, InvalidOperation

transaction_rows = [
    "SKU-1,4,25.00",
    "SKU-2,2,49.99",
    "SKU-3,3,40.00",
    "SKU-4,x,10.00",
    "SKU-5,5,bad",
    "SKU-6,-1,200.00",
    "SKU-7,1,100.00",
]

def parse_positive_int(text: str):
    try:
        value = int(text)
    except ValueError:
        return None
    return value if value > 0 else None

def parse_positive_decimal(text: str):
    try:
        value = Decimal(text)
    except InvalidOperation:
        return None
    return value if value > 0 else None

### Readable solution

In [29]:
qualified_transactions = []

for row in transaction_rows:
    sku, raw_quantity, raw_price = row.split(",")

    if (quantity := parse_positive_int(raw_quantity)) is None:
        continue
    if (price := parse_positive_decimal(raw_price)) is None:
        continue
    if (total := quantity * price) < Decimal("100"):
        continue

    qualified_transactions.append(
        {
            "sku": sku,
            "quantity": quantity,
            "price": price,
            "total": total,
        }
    )

qualified_transactions

[{'sku': 'SKU-1',
  'quantity': 4,
  'price': Decimal('25.00'),
  'total': Decimal('100.00')},
 {'sku': 'SKU-3',
  'quantity': 3,
  'price': Decimal('40.00'),
  'total': Decimal('120.00')},
 {'sku': 'SKU-7',
  'quantity': 1,
  'price': Decimal('100.00'),
  'total': Decimal('100.00')}]

In [30]:
assert qualified_transactions == [
    {
        "sku": "SKU-1",
        "quantity": 4,
        "price": Decimal("25.00"),
        "total": Decimal("100.00"),
    },
    {
        "sku": "SKU-3",
        "quantity": 3,
        "price": Decimal("40.00"),
        "total": Decimal("120.00"),
    },
    {
        "sku": "SKU-7",
        "quantity": 1,
        "price": Decimal("100.00"),
        "total": Decimal("100.00"),
    },
]

Do not force multi-stage validation into one dense comprehension. The ordinary loop gives each rejection condition a clear location.

# Problem 11 — `any` and captured evidence

Detect whether any response exceeds 250 ms and capture the first violating response. Then examine a subtle failure mode.

In [31]:
responses = [
    {"server": "edge-1", "latency_ms": 120},
    {"server": "edge-2", "latency_ms": 180},
    {"server": "edge-3", "latency_ms": 310},
    {"server": "edge-4", "latency_ms": 90},
]

### Assignment-expression version

In [32]:
violating_response = None

has_violation = any(
    (violating_response := response)["latency_ms"] > 250
    for response in responses
)

print(has_violation, violating_response)

assert has_violation is True
assert violating_response == {"server": "edge-3", "latency_ms": 310}

True {'server': 'edge-3', 'latency_ms': 310}


When no item matches, `violating_response` becomes the last examined item, not `None`. For first-match retrieval, `next` is clearer.

In [33]:
first_violation = next(
    (response for response in responses if response["latency_ms"] > 250),
    None,
)

assert first_violation == {"server": "edge-3", "latency_ms": 310}
first_violation

{'server': 'edge-3', 'latency_ms': 310}

# Problem 12 — Count evaluations in three designs

Compare:

1. duplicated call in a comprehension;
2. ordinary loop with a temporary variable;
3. comprehension using `:=`.

All outputs must match.

In [34]:
def make_counted_transform():
    state = {"calls": 0}

    def transform(value: int) -> int:
        state["calls"] += 1
        return (value * value + 3 * value + 7) % 97

    return transform, state

inputs = list(range(50))

In [35]:
transform, state = make_counted_transform()
version_1 = [
    transform(value)
    for value in inputs
    if transform(value) % 5 == 0
]
calls_1 = state["calls"]

transform, state = make_counted_transform()
version_2 = []
for value in inputs:
    transformed = transform(value)
    if transformed % 5 == 0:
        version_2.append(transformed)
calls_2 = state["calls"]

transform, state = make_counted_transform()
version_3 = [
    transformed
    for value in inputs
    if (transformed := transform(value)) % 5 == 0
]
calls_3 = state["calls"]

print("calls:", calls_1, calls_2, calls_3)
print("selected:", len(version_3))

calls: 61 50 50
selected: 11


In [36]:
assert version_1 == version_2 == version_3
assert calls_2 == calls_3 == len(inputs)
assert calls_1 == len(inputs) + len(version_1)

The walrus version combines the compact shape of a comprehension with the single-evaluation behavior of the loop.

# Problem 13 — Refactor an anti-pattern

Review:

```python
if (a := get_a()) and (b := get_b(a)) and (c := normalize(b)) and (d := validate(c)):
    publish(d)
```

Refactor it to make failure stages visible.

In [37]:
def get_a():
    return {"raw": " 42 "}

def get_b(a):
    return a.get("raw")

def normalize(value):
    return value.strip() if isinstance(value, str) else None

def validate(value):
    return int(value) if value and value.isdigit() else None

published = []

def publish(value):
    published.append(value)

### Solution

In [38]:
published.clear()

source = get_a()
if source is None:
    status = "missing source"
elif (raw_value := get_b(source)) is None:
    status = "missing raw value"
elif (normalized := normalize(raw_value)) is None:
    status = "normalization failed"
elif (validated := validate(normalized)) is None:
    status = "validation failed"
else:
    publish(validated)
    status = "published"

print(status, published)
assert status == "published"
assert published == [42]

published [42]


A chain of `elif` clauses gives every failure mode a named location. The remaining assignment expressions are short and local.

# Problem 14 — Repeated regex extraction

Extract every `key=value` pair from a configuration string. Advance the search position after each match.

In [39]:
pair_pattern = re.compile(r"(?P<key>[A-Za-z_]\w*)=(?P<value>[^;\s]+)")
config_text = "host=db01; port=5432; ssl=true; retry_count=4"

### Solution

In [40]:
pairs = []
position = 0

while match := pair_pattern.search(config_text, position):
    pairs.append((match["key"], match["value"]))
    position = match.end()

pairs

[('host', 'db01'), ('port', '5432'), ('ssl', 'true'), ('retry_count', '4')]

In [41]:
assert pairs == [
    ("host", "db01"),
    ("port", "5432"),
    ("ssl", "true"),
    ("retry_count", "4"),
]

This is safe because the pattern consumes characters, so `match.end()` advances. With a zero-length-capable pattern, guarantee progress explicitly.

# Problem 15 — Nested lookup without repeated indexing

Return labels only for orders with a non-empty customer email. Avoid repeating `.get(...)`, but keep the condition readable.

In [42]:
orders = [
    {"id": 101, "customer": {"email": "ana@example.test"}},
    {"id": 102, "customer": None},
    {"id": 103},
    {"id": 104, "customer": {"email": ""}},
    {"id": 105, "customer": {"email": "mira@example.test"}},
]

### Solution

In [43]:
labels = [
    f"order={order['id']} email={email}"
    for order in orders
    if (customer := order.get("customer")) is not None
    and (email := customer.get("email"))
]

labels

['order=101 email=ana@example.test', 'order=105 email=mira@example.test']

In [44]:
assert labels == [
    "order=101 email=ana@example.test",
    "order=105 email=mira@example.test",
]

The final truthiness check intentionally rejects `""`. If empty strings are valid, use `is not None` instead.

# Problem 16 — Mutable objects and shared references

Predict whether `left` and `right` refer to the same list:

```python
left = (right := [1, 2] + [3, 4])
```

In [45]:
left = (right := [1, 2] + [3, 4])

print("same object:", left is right)
right.append(5)
print("left:", left)
print("right:", right)

same object: True
left: [1, 2, 3, 4, 5]
right: [1, 2, 3, 4, 5]


In [46]:
assert left is right
assert left == right == [1, 2, 3, 4, 5]

One list is created. Both names reference it. Assignment expressions do not copy mutable objects.

# Problem 17 — Build a safe batching loop

Consume items in batches of at most three. Stop cleanly at exhaustion.

In [47]:
from itertools import islice

def take_batch(iterator, size):
    batch = list(islice(iterator, size))
    return batch or None

source_items = iter(range(10))
batches = []

### Solution

In [48]:
while batch := take_batch(source_items, 3):
    batches.append(batch)

batches

[[0, 1, 2], [3, 4, 5], [6, 7, 8], [9]]

In [49]:
assert batches == [
    [0, 1, 2],
    [3, 4, 5],
    [6, 7, 8],
    [9],
]

Document the helper's contract clearly, especially if an empty collection could be valid data rather than an exhaustion signal.

# Problem 18 — Capstone telemetry pipeline

Process newline-delimited JSON telemetry:

1. read one line at a time;
2. stop at end-of-stream;
3. parse JSON objects safely;
4. validate sensor, unit, and temperature;
5. keep readings at or above 70 °C;
6. retain the highest alert per sensor;
7. record rejected rows with reasons.

In [50]:
from io import StringIO

telemetry_text = (
    '{"sensor": "A1", "temperature": 72.5, "unit": "C"}\n'
    '{"sensor": "B2", "temperature": 49.0, "unit": "C"}\n'
    'not-json\n'
    '{"sensor": "A1", "temperature": 75.0, "unit": "C"}\n'
    '{"sensor": "", "temperature": 90, "unit": "C"}\n'
    '{"sensor": "C3", "temperature": "81.2", "unit": "C"}\n'
    '{"sensor": "D4", "temperature": null, "unit": "C"}\n'
    '{"sensor": "E5", "temperature": 80, "unit": "F"}\n'
)

ALERT_THRESHOLD_C = 70.0

def safe_json_object(text: str):
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

def to_float(value):
    if isinstance(value, bool):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

### Complete solution

In [51]:
stream = StringIO(telemetry_text)
highest_alert_by_sensor = {}
rejected = []
line_number = 0

while line := stream.readline():
    line_number += 1
    stripped = line.strip()

    if not stripped:
        continue

    if (obj := safe_json_object(stripped)) is None:
        rejected.append((line_number, "invalid JSON object", stripped))
        continue

    if not (sensor := obj.get("sensor")):
        rejected.append((line_number, "missing sensor", obj))
        continue

    if obj.get("unit") != "C":
        rejected.append((line_number, "unsupported unit", obj))
        continue

    if (temperature := to_float(obj.get("temperature"))) is None:
        rejected.append((line_number, "invalid temperature", obj))
        continue

    if temperature < ALERT_THRESHOLD_C:
        continue

    previous = highest_alert_by_sensor.get(sensor)
    if previous is None or temperature > previous["temperature"]:
        highest_alert_by_sensor[sensor] = {
            "sensor": sensor,
            "temperature": temperature,
            "line_number": line_number,
        }

alerts = sorted(
    highest_alert_by_sensor.values(),
    key=lambda item: (-item["temperature"], item["sensor"]),
)

print("Alerts:")
for alert in alerts:
    print(alert)

print("\nRejected:")
for item in rejected:
    print(item)

Alerts:
{'sensor': 'C3', 'temperature': 81.2, 'line_number': 6}
{'sensor': 'A1', 'temperature': 75.0, 'line_number': 4}

Rejected:
(3, 'invalid JSON object', 'not-json')
(5, 'missing sensor', {'sensor': '', 'temperature': 90, 'unit': 'C'})
(7, 'invalid temperature', {'sensor': 'D4', 'temperature': None, 'unit': 'C'})
(8, 'unsupported unit', {'sensor': 'E5', 'temperature': 80, 'unit': 'F'})


In [52]:
assert alerts == [
    {"sensor": "C3", "temperature": 81.2, "line_number": 6},
    {"sensor": "A1", "temperature": 75.0, "line_number": 4},
]

assert [reason for _, reason, _ in rejected] == [
    "invalid JSON object",
    "missing sensor",
    "invalid temperature",
    "unsupported unit",
]

### Capstone review

Good uses of `:=`:

- `while line := stream.readline()` — fetch and test;
- `(obj := safe_json_object(...))` — parse once and reuse;
- `(sensor := obj.get("sensor"))` — retrieve once and validate;
- `(temperature := to_float(...))` — convert once and reuse.

Ordinary assignments remain clearer for values used across several statements, such as `stripped` and `previous`.

# Additional challenge set

Complete each problem before studying the supplied solution.

## Challenge A — Header reader

Read lines until the first blank line. A line containing only spaces counts as blank.

In [53]:
header_stream = StringIO("Host: example.test\nAccept: application/json\n   \nBODY")
headers = []

while (line := header_stream.readline()) and (stripped := line.strip()):
    headers.append(stripped)

assert headers == ["Host: example.test", "Accept: application/json"]
headers

['Host: example.test', 'Accept: application/json']

## Challenge B — First successful parser

Try parser functions in order and stop at the first non-`None` result. The value `0` is valid.

In [54]:
def parse_hex(text):
    try:
        return int(text, 16) if text.lower().startswith("0x") else None
    except ValueError:
        return None

def parse_decimal(text):
    try:
        return int(text)
    except ValueError:
        return None

def parse_word_zero(text):
    return 0 if text.strip().lower() == "zero" else None

parsers = [parse_hex, parse_decimal, parse_word_zero]
candidate = "zero"

parsed = None
for parser in parsers:
    if (value := parser(candidate)) is not None:
        parsed = value
        break

assert parsed == 0
parsed

0

## Challenge C — Valid integers and squares

Strip, parse once, keep values whose square is below 100, and return `(value, square)`.

In [55]:
raw_numbers = [" 7 ", "-8", "10", "x", "+3", " -12 ", "0"]

def parse_int(text):
    try:
        return int(text.strip())
    except ValueError:
        return None

small_squares = [
    (number, square)
    for raw in raw_numbers
    if (number := parse_int(raw)) is not None
    and (square := number * number) < 100
]

assert small_squares == [(7, 49), (-8, 64), (3, 9), (0, 0)]
small_squares

[(7, 49), (-8, 64), (3, 9), (0, 0)]

## Challenge D — First duplicate

Return the first repeated item or `None`. A normal loop is best because set mutation inside a dense expression would obscure intent.

In [56]:
def first_duplicate(values):
    seen = set()

    for value in values:
        if value in seen:
            return value
        seen.add(value)

    return None

assert first_duplicate([3, 1, 4, 1, 5]) == 1
assert first_duplicate([1, 2, 3]) is None

## Challenge E — Command loop simulation

Process commands until `"quit"`. Ignore blank commands and normalize once.

In [57]:
commands = iter(["  STATUS ", "", "Run", " QUIT ", "ignored"])
processed = []
_END = object()

while (raw := next(commands, _END)) is not _END:
    if not (command := raw.strip().lower()):
        continue
    if command == "quit":
        break
    processed.append(command)

assert processed == ["status", "run"]
processed

['status', 'run']

## Challenge F — Find the first regex match across documents

Search documents in order and stop at the first matching document. Return both the document ID and captured ticket number.

In [58]:
ticket_pattern = re.compile(r"\bTICKET-(\d{4})\b")
documents = [
    {"id": "doc-1", "text": "No reference here."},
    {"id": "doc-2", "text": "Investigate TICKET-4821 immediately."},
    {"id": "doc-3", "text": "Later mention TICKET-9999."},
]

found = None
for document in documents:
    if match := ticket_pattern.search(document["text"]):
        found = {
            "document_id": document["id"],
            "ticket": match.group(1),
        }
        break

assert found == {"document_id": "doc-2", "ticket": "4821"}
found

{'document_id': 'doc-2', 'ticket': '4821'}

## Challenge G — Running input validation

Consume strings until exhaustion. Keep parsed non-negative integers and collect rejected raw values.

In [59]:
raw_values = iter(["10", " 0 ", "-3", "x", "7"])
valid_values = []
invalid_values = []
_END = object()

while (raw := next(raw_values, _END)) is not _END:
    if (number := parse_int(raw)) is None or number < 0:
        invalid_values.append(raw)
    else:
        valid_values.append(number)

assert valid_values == [10, 0, 7]
assert invalid_values == ["-3", "x"]

# Testing semantic equivalence

A refactor involving `:=` should preserve:

- returned values;
- number and order of side-effecting calls;
- exception behavior;
- treatment of falsy values;
- stopping conditions;
- scope expectations.

In [60]:
def conventional(values):
    output = []
    for value in values:
        transformed = value.strip().upper()
        if transformed.startswith("A"):
            output.append(transformed)
    return output

def with_assignment_expression(values):
    return [
        transformed
        for value in values
        if (transformed := value.strip().upper()).startswith("A")
    ]

test_cases = [
    [],
    [""],
    ["alpha", " beta ", "ALPACA"],
    [" a ", "A", "z", "aardvark"],
]

for case in test_cases:
    assert conventional(case) == with_assignment_expression(case)

print("All semantic-equivalence tests passed.")

All semantic-equivalence tests passed.


# Final code-review rubric

A production-quality use of `:=` should usually satisfy:

- **Single evaluation:** an operation is no longer repeated.
- **Immediate reuse:** the assigned value is consumed nearby.
- **Descriptive name:** the binding explains the value.
- **Visible control flow:** readers can identify when evaluation occurs.
- **Correct falsy handling:** `0`, `False`, `""`, and empty containers are intentional.
- **Safe scope:** post-comprehension bindings do not create surprises.
- **Explicit precedence:** parentheses communicate the assignment boundary.
- **Limited density:** one or two simple assignment expressions are usually enough.
- **Tests:** equivalence and call counts are verified.

A normal assignment is not inferior. Choose the form that makes correctness easiest to see.

# Summary

The strongest recurring patterns are:

```python
# Fetch-and-test loop
while chunk := stream.read(size):
    process(chunk)

# Compute once in a comprehension filter
results = [
    transformed
    for item in items
    if (transformed := transform(item)) is not None
]

# Match-and-reuse condition
if match := pattern.search(text):
    use(match)
```

The objective is not fewer lines at any cost. The objective is clear production, testing, and immediate reuse of a value without duplicate work.